# BioRAG-Hybrid — Official BioASQ11 Ingestion Pipeline

This notebook uses only the official BioASQ11 questions and the aligned PubMed abstract corpus. It validates the release, checks provenance, verifies the question split, and confirms that the searchable corpus does not contain BioASQ gold snippets.

Run this notebook before the retrieval notebook.

In [1]:
from pathlib import Path
import hashlib, json, re, sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

QUESTION_FILE = ROOT / 'data/official/training11b.json'
CORPUS_FILE = ROOT / 'data/official/pubmed_abstracts.json'
SPLIT_FILE = ROOT / 'data/official/question_split.json'
META_FILE = ROOT / 'data/official/pubmed_abstracts.metadata.json'
print('Repository:', ROOT)
print('BioASQ file:', QUESTION_FILE.exists())
print('PubMed corpus:', CORPUS_FILE.exists())

Repository: /home/anmol/BioRAG-Hybrid-Evaluation1
BioASQ file: True
PubMed corpus: True


In [2]:
# Official BioASQ11 release inspection
raw = json.loads(QUESTION_FILE.read_text(encoding='utf-8'))
questions = raw['questions']
assert len(questions) == 4719, f'Unexpected BioASQ record count: {len(questions)}'
ids = [q['id'] for q in questions]
assert len(ids) == len(set(ids)), 'Duplicate question IDs detected'
type_counts = {}
for q in questions:
    type_counts[q.get('type', 'unknown')] = type_counts.get(q.get('type', 'unknown'), 0) + 1
document_refs = sum(len(q.get('documents', [])) for q in questions)
snippet_count = sum(len(q.get('snippets', [])) for q in questions)
print('Questions:', len(questions))
print('Question types:', type_counts)
print('Document references:', document_refs)
print('Gold snippets:', snippet_count)
print('BioASQ SHA-256:', hashlib.sha256(QUESTION_FILE.read_bytes()).hexdigest())

Questions: 4719
Question types: {'summary': 1130, 'list': 901, 'yesno': 1271, 'factoid': 1417}
Document references: 42608
Gold snippets: 56795
BioASQ SHA-256: 6df656862ca860efc355c7805d07ddca700d64ecc3785c519a49afccaaeeac98


## Build step

The following cell is disabled because the official 40k-ID download has already been completed. Set `RUN_PUBMED_DOWNLOAD = True` only when rebuilding from the official BioASQ release. The script uses NCBI E-utilities, retry/backoff, checkpointing, and writes provenance metadata.

In [3]:
import subprocess
RUN_PUBMED_DOWNLOAD = False
if RUN_PUBMED_DOWNLOAD:
    subprocess.run([
        sys.executable, str(ROOT / 'tools/build_official_corpus.py'),
        '--questions', str(QUESTION_FILE),
        '--output', str(CORPUS_FILE),
        '--split-output', str(SPLIT_FILE),
        '--email', 'YOUR_EMAIL@example.com'
    ], check=True)
else:
    print('Using existing official PubMed corpus; no download performed.')

Using existing official PubMed corpus; no download performed.


In [4]:
# Validate the generated official searchable corpus and provenance
assert CORPUS_FILE.exists(), 'Run the build cell first; official corpus is missing.'
corpus = json.loads(CORPUS_FILE.read_text(encoding='utf-8'))
metadata = json.loads(META_FILE.read_text(encoding='utf-8'))
assert len(corpus) == metadata['retrieved_documents']
assert metadata['gold_snippets_indexed'] is False
assert hashlib.sha256(CORPUS_FILE.read_bytes()).hexdigest() == metadata['corpus_sha256']
assert all(str(d['id']).isdigit() for d in corpus)
assert all(d.get('title') or d.get('text') for d in corpus)
print('Retrieved PubMed abstracts:', len(corpus))
print('Requested PubMed IDs:', metadata['requested_pubmed_ids'])
print('Unavailable references:', metadata['requested_pubmed_ids'] - len(corpus))
print('Corpus SHA-256:', metadata['corpus_sha256'])
print('Gold snippets indexed:', metadata['gold_snippets_indexed'])
print('Example document:', {k: corpus[0][k] for k in ('id', 'title')})

Retrieved PubMed abstracts: 40160
Requested PubMed IDs: 40221
Unavailable references: 61
Corpus SHA-256: ec08b2c85da78c94d06a0b01b21b50c2d7d4ddce231edc7811b3062cc0a8cdb6
Gold snippets indexed: False
Example document: {'id': '9797', 'title': 'Morphological, cytological and biological observations on viruses isolated from patients with subacute thyroiditis de Quervain.'}


In [5]:
# Validate leakage-safe question-ID split
splits = json.loads(SPLIT_FILE.read_text(encoding='utf-8'))
train, validation, test = (set(splits[k]) for k in ('train', 'validation', 'test'))
assert not (train & validation or train & test or validation & test)
assert train | validation | test == set(ids)
print('Train:', len(train))
print('Validation:', len(validation))
print('Test:', len(test))
print('Question-ID split is disjoint and complete: PASS')

Train: 3712
Validation: 506
Test: 501
Question-ID split is disjoint and complete: PASS


## Ingestion conclusion

The official ingestion stage is complete. BioASQ supplies questions and gold evaluation fields; PubMed supplies searchable title/abstract text. The two are aligned by PMID, and gold snippets remain evaluation labels only.